# One-hot Encoding 🔥

Demonstration of One-hot encoding (Not a complete solution)

In [2]:
import pandas as pd
import re
from collections import defaultdict


## We can use NLP methods to extract the chemical elements and assign them 1s and 0s
However, we have to be very careful here, NLP methods are not meant for materials specific applications- so we need to double check and understand exactly what is going on.

In [12]:
data=pd.read_csv("5_Thermal_Cond.csv")
data.shape


(58, 2)

In [13]:
# Function to extract unique elements from a chemical formula
def extract_elements(formula):
    return re.findall(r'[A-Z][a-z]?', formula)

    #re.findall(pattern, string)
#Searches string for all occurrences matching the pattern

#Returns them as a list



r''	Raw string — tells Python not to interpret \ as escape characters

[A-Z]	Match a single uppercase letter (e.g. C, H, O, N, S)

[a-z]?	Match an optional lowercase letter (e.g. l in Al, r in Br)

[A-Z][a-z]?	Together, matches one chemical element symbol (like C, Al, Fe)

In [15]:
# Create a full set of elements
all_elements = set()
for formula in data['Compound']:   #For each compound in the data
    all_elements.update(extract_elements(formula))  # call the function to extract chemical elements
all_elements = sorted(all_elements) 
#sorted() gives you a consistent order of elements (alphabetical)
#This makes sure your one-hot encoded columns always appear in the same order 


In [16]:
# Now we turn the list of elements into One-hot encode
def one_hot_formula(formula, elements):
    present = extract_elements(formula)
    return [1 if el in present else 0 for el in elements]

In [9]:
# Apply encoding
element_columns = [f'has_{el}' for el in all_elements]
one_hot_encoded = data['Compound'].apply(lambda f: one_hot_formula(f, all_elements))
one_hot_df = pd.DataFrame(one_hot_encoded.tolist(), columns=element_columns)

# Combine with original dataframe
df_encoded = pd.concat([data, one_hot_df], axis=1)
print(df_encoded)

    Compound  K (W/mK)  has_Al  has_Ar  has_As  has_B  has_Ba  has_Be  has_Br  \
0     C12H24    0.1400       0       0       0      0       0       0       0   
1      C3H6O    0.1535       0       0       0      0       0       0       0   
2       C2H2    0.0218       0       0       0      0       0       0       0   
3         Al  255.9400       1       0       0      0       0       0       0   
4      Al2O3    0.0420       1       0       0      0       0       0       0   
5        NH3    0.0252       0       0       0      0       0       0       0   
6      NH4Br    2.5100       0       0       0      0       0       0       1   
7      NH4Cl    2.5100       0       0       0      0       0       0       0   
8         Sb   18.4100       0       0       0      0       0       0       0   
9    C6H5NH2    0.1700       0       0       0      0       0       0       0   
10    Sb2Te3    3.3050       0       0       0      0       0       0       0   
11        Ar    0.0180      

In [10]:
df_encoded.shape

(58, 31)

In [17]:
#Note that the elements are not normalized.
#So, C6H5NH2 will have a total of 3 elements, where Al has 1 : this is not right
#So need to normalize each row - use a lambda function to do this

# -----------
# Pymatgen  💪
A better way to handle and extract elemental features in a chemically aware way.

In [21]:
import pandas as pd
from pymatgen.core.composition import Composition


In [28]:
#data=pd.read_csv("5_Thermal_Cond.csv")
#data.shape



In [33]:
# Step 1: Extract all unique elements across all formulas
all_elements_pm = set()
for formula in data['Compound']:
    comp = Composition(formula)
    all_elements_pm.update(comp.elements)



In [34]:
# Convert to sorted element symbols
all_elements_pm = sorted([el.symbol for el in all_elements_pm])


In [35]:
# Step 2: Create one-hot encoding
def one_hot_pymatgen(formula, elements):
    comp = Composition(formula)
    present = [el.symbol for el in comp.elements]
    return [1 if el in present else 0 for el in elements]



In [36]:
# Apply the one-hot encoding
one_hot_cols = [f'has_{el}' for el in all_elements_pm]
one_hot_encoded_pm = df['Compound'].apply(lambda x: one_hot_pymatgen(x, all_elements_pm))
one_hot_df_pm = pd.DataFrame(one_hot_encoded_pm.tolist(), columns=one_hot_cols)

# Combine with original
df_encoded_pm = pd.concat([df, one_hot_df_pm], axis=1)
print(df_encoded_pm)

    Compound  K (W/mK)  has_Al  has_Ar  has_As  has_B  has_Ba  has_Be  has_Br  \
0     C12H24    0.1400       0       0       0      0       0       0       0   
1      C3H6O    0.1535       0       0       0      0       0       0       0   
2       C2H2    0.0218       0       0       0      0       0       0       0   
3         Al  255.9400       1       0       0      0       0       0       0   
4      Al2O3    0.0420       1       0       0      0       0       0       0   
5        NH3    0.0252       0       0       0      0       0       0       0   
6      NH4Br    2.5100       0       0       0      0       0       0       1   
7      NH4Cl    2.5100       0       0       0      0       0       0       0   
8         Sb   18.4100       0       0       0      0       0       0       0   
9    C6H5NH2    0.1700       0       0       0      0       0       0       0   
10    Sb2Te3    3.3050       0       0       0      0       0       0       0   
11        Ar    0.0180      

In [ ]:
df_encoded